# Task 1: Live Data Collection
## AI CEO Strategic Intelligence Agent — Lufthansa

This notebook collects live, real information about Lufthansa 

### Minimum Requirements
- At least 100 documents
- Automatic collection process

In [1]:

import json
import os
from datetime import datetime
from ddgs import DDGS
import wikipediaapi



## Source 1: DuckDuckGo Live Search

DuckDuckGo (DDGS) lets us search the web in real time without any API key.

We define search queries covering all 5 required categories:
- **Company** — press releases, investor relations, CEO strategy
- **News** — financial news, industry news, technology news
- **Market** — competitors, fleet, acquisitions, industry reports
- **Community** — passenger reviews, strikes, loyalty program
- **Research** — sustainability, AI, innovation

Each query returns real, live web results at the time of running.

In [2]:
# Search queries in English AND German to get richer multilingual data
# Lufthansa is German, so German news sources have unique content
search_queries = [
    # English queries
    {"query": "Lufthansa press release 2025 2026",        "category": "company"},
    {"query": "Lufthansa investor relations earnings",     "category": "company"},
    {"query": "Lufthansa CEO strategy announcement",       "category": "company"},
    {"query": "Lufthansa financial news revenue 2026",     "category": "news"},
    {"query": "Lufthansa airline industry news 2026",      "category": "news"},
    {"query": "Lufthansa technology digital innovation",   "category": "news"},
    {"query": "Lufthansa competitor Ryanair Air France",   "category": "market"},
    {"query": "Lufthansa ITA Airways acquisition",         "category": "market"},
    {"query": "Lufthansa fleet Airbus Boeing order",       "category": "market"},
    {"query": "Lufthansa passenger review experience",     "category": "community"},
    {"query": "Lufthansa pilot strike union 2025 2026",    "category": "community"},
    {"query": "Lufthansa sustainability SAF emissions",    "category": "research"},
    {"query": "Lufthansa AI innovation technology",        "category": "research"},

    # German queries
    {"query": "Lufthansa Pressemitteilung 2025 2026",       "category": "company"},
    {"query": "Lufthansa Geschäftsbericht Gewinn",          "category": "company"},
    {"query": "Lufthansa Konkurrenz Wettbewerb",            "category": "market"},
    {"query": "Lufthansa Pilotenstreik Gewerkschaft",       "category": "community"},
    {"query": "Lufthansa Nachhaltigkeit Klimaschutz",       "category": "research"},
]

print(f"Total queries: {len(search_queries)}")

Total queries: 18


In [3]:
# Collect live search results from DuckDuckGo
ddgs_documents = []

with DDGS() as ddgs:
    for item in search_queries:

        results = list(ddgs.text(item["query"], max_results=10))

        for r in results:
            ddgs_documents.append({
                "title"        : r.get("title", ""),
                "url"          : r.get("href", ""),
                "summary"      : r.get("body", ""),
                "category"     : item["category"],
                "source"       : "DuckDuckGo Search",
                "date"         : datetime.now().strftime("%Y-%m-%d"),
                "collected_at" : datetime.now().isoformat()
            })

        print(f"[{item['category']}] {item['query']} → {len(results)} results")

print(f"\nTotal documents collected: {len(ddgs_documents)}")

[company] Lufthansa press release 2025 2026 → 10 results
[company] Lufthansa investor relations earnings → 10 results
[company] Lufthansa CEO strategy announcement → 10 results
[news] Lufthansa financial news revenue 2026 → 10 results
[news] Lufthansa airline industry news 2026 → 10 results
[news] Lufthansa technology digital innovation → 10 results
[market] Lufthansa competitor Ryanair Air France → 10 results
[market] Lufthansa ITA Airways acquisition → 10 results
[market] Lufthansa fleet Airbus Boeing order → 10 results
[community] Lufthansa passenger review experience → 10 results
[community] Lufthansa pilot strike union 2025 2026 → 10 results
[research] Lufthansa sustainability SAF emissions → 10 results
[research] Lufthansa AI innovation technology → 10 results
[company] Lufthansa Pressemitteilung 2025 2026 → 10 results
[company] Lufthansa Geschäftsbericht Gewinn → 10 results
[market] Lufthansa Konkurrenz Wettbewerb → 10 results
[community] Lufthansa Pilotenstreik Gewerkschaft → 1

## Step 2: Collect from Wikipedia

Wikipedia gives us structured, reliable background information about 
Lufthansa and its subsidiaries. This is our second independent source.

In [4]:
# Wikipedia topics to collect about Lufthansa and subsidiaries
wiki_topics = [
    "Lufthansa",
    "Lufthansa Group",
    "Eurowings",
    "Austrian Airlines",
    "SWISS International Air Lines",
    "Brussels Airlines",
    "Lufthansa Cargo",
    "Lufthansa Technik",
    "Miles & More",
    "Star Alliance"
]

# We collect from both English and German Wikipedia
# Lufthansa is a German company so German Wikipedia has rich detail
languages = ["en", "de"]

wiki_documents = []

for lang in languages:

    wiki = wikipediaapi.Wikipedia(
        language=lang,
        user_agent="LufthansaIntelligenceAgent/1.0"
    )

    print(f"\nCollecting from {lang.upper()} Wikipedia:")

    for topic in wiki_topics:
        page = wiki.page(topic)

        if page.exists():
            wiki_documents.append({
                "title"        : page.title,
                "url"          : page.fullurl,
                "summary"      : page.summary[:500],
                "category"     : "research",
                "source"       : f"Wikipedia ({lang.upper()})",
                "date"         : datetime.now().strftime("%Y-%m-%d"),
                "language"     : lang,
                "collected_at" : datetime.now().isoformat()
            })
            print(f"  Collected: {page.title}")
        else:
            print(f"  Not found: {topic}")

print(f"\nTotal Wikipedia documents: {len(wiki_documents)}")


  Collected: Lufthansa
  Collected: Lufthansa
  Collected: Eurowings
  Collected: Austrian Airlines
  Collected: Swiss International Air Lines
  Collected: Brussels Airlines
  Collected: Lufthansa Cargo
  Collected: Lufthansa Technik
  Collected: Miles & More
  Collected: Star Alliance

  Collected: Lufthansa
  Collected: Lufthansa Group
  Collected: Eurowings
  Collected: Austrian Airlines
  Not found: SWISS International Air Lines
  Collected: Brussels Airlines
  Collected: Lufthansa Cargo
  Collected: Lufthansa Technik
  Collected: Miles & More
  Collected: Star Alliance

Total Wikipedia documents: 19


## Step 3: Collect from RSS Feeds (3rd Independent Source)

We use RSS feeds from aviation news websites.
RSS feeds are free, public, and give us real live articles.

In [5]:
import feedparser
from bs4 import BeautifulSoup

rss_feeds = [
    # News/Market sources - targeted Google News search (high yield, all relevant)
    {"url": "https://news.google.com/rss/search?q=Lufthansa+when:30d&hl=en-US&gl=US&ceid=US:en",
     "source": "Google News", "category": "news"},
    {"url": "https://news.google.com/rss/search?q=Lufthansa+Group+strategy&hl=en-US&gl=US&ceid=US:en",
     "source": "Google News Strategy", "category": "market"},
    {"url": "https://news.google.com/rss/search?q=Lufthansa+financial+results&hl=en-US&gl=US&ceid=US:en",
     "source": "Google News Finance", "category": "news"},

    # Community source - Reddit, genuinely Lufthansa-focused subreddit
    {"url": "https://www.reddit.com/r/lufthansa/.rss",
     "source": "Reddit Community", "category": "community"},
]

rss_documents = []

for feed_info in rss_feeds:
    feed = feedparser.parse(feed_info["url"], agent="LufthansaCEOIntelligenceAgent/1.0")
    print(f"{feed_info['source']} → {len(feed.entries)} items fetched")

    for entry in feed.entries:
        title   = entry.get("title", "")
        url     = entry.get("link", "")
        summary = entry.get("summary", "") or entry.get("description", "")
        date    = entry.get("published", datetime.now().strftime("%Y-%m-%d"))

        summary = BeautifulSoup(summary, "html.parser").get_text()[:500] if summary else ""

        rss_documents.append({
            "title"        : title,
            "url"          : url,
            "summary"      : summary,
            "category"     : feed_info["category"],
            "source"       : feed_info["source"],
            "date"         : date,
            "collected_at" : datetime.now().isoformat()
        })

print(f"\nTotal RSS documents collected: {len(rss_documents)}")

Google News → 100 items fetched
Google News Strategy → 80 items fetched
Google News Finance → 89 items fetched
Reddit Community → 25 items fetched

Total RSS documents collected: 294


In [6]:
all_documents = ddgs_documents + wiki_documents + rss_documents

os.makedirs("../data", exist_ok=True)
output_path = "../data/collected_documents.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_documents, f, indent=2, ensure_ascii=False)

print(f"Total raw documents saved : {len(all_documents)}")
print(f"  DuckDuckGo : {len(ddgs_documents)}")
print(f"  Wikipedia  : {len(wiki_documents)}")
print(f"  RSS Feeds  : {len(rss_documents)}")
print(f"\nSaved to: {output_path}")

Total raw documents saved : 493
  DuckDuckGo : 180
  Wikipedia  : 19
  RSS Feeds  : 294

Saved to: ../data/collected_documents.json
